In [0]:
!pip install -r /Workspace/Repos/operacion-maker/pacifico-metadata-ai/requirements.txt

In [0]:
import os
import time
import mlflow
from mlflow.models import infer_signature
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()
from agent import MetadataGovernanceAgent

In [0]:
mlflow.set_registry_uri("databricks-uc")
# Define tu catálogo y esquema
catalog_name = "workspace"  # <-- Cambia esto si usas otro catálogo
schema_name = "default"  # <-- Cambia esto si usas otro esquema
model_name = f"{catalog_name}.{schema_name}.metabuilder_agent"
endpoint_name = "metabuilder-endpoint"

print(f"Model Name: {model_name}")
print(f"Endpoint Name: {endpoint_name}")

In [0]:
# 1. Definir el esquema base para el agente
input_example = {
    "messages": [{"role": "user", "content": "main.default.my_table"}],
    "custom_inputs": {"thread_id": "12345"},
}

# 2. Inferir la firma explícitamente usando el ejemplo y un output de texto
signature = infer_signature(
    model_input=input_example,
    model_output=["Respuesta generada por el agente"]
)

with mlflow.start_run() as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent_model",
        python_model="agent.py", 
        code_path=[
            "agent.py", "graph", "state", "nodes", 
            "prompts", "tools", "config", "context", "routers"
        ],
        pip_requirements="requirements.txt", 
        input_example=input_example,
        signature=signature
    )

print(f"Modelo registrado localmente en: {model_info.model_uri}")

# Registrar el modelo en Unity Catalog
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri, name=model_name
)

print(f"✅ Modelo {model_name} versión {registered_model.version} registrado en Unity Catalog.")

In [0]:
from databricks import agents

print(f"Desplegando el Agente de Gobierno en el Endpoint: {endpoint_name}...")

# databricks.agents.deploy() orquesta el Serving + Review App + Agent Evaluation
try:
    deployment_info = agents.deploy(
        model_name=model_name,
        model_version=registered_model.version,
        endpoint_name=endpoint_name,
        scale_to_zero=True,
        # Inyección de Secretos estilo "Agent Framework"
        environment_vars={
            "DATABRICKS_HOST": "{{secrets/metadatos_scope/db_host}}",
            "DATABRICKS_TOKEN": "{{secrets/metadatos_scope/db_token}}",
            "DATABRICKS_SQL_WAREHOUSE_ID": "{{secrets/metadatos_scope/db_warehouse_id}}"
        }
    )
    print(f"✅ Agente '{endpoint_name}' desplegado exitosamente.")
except Exception as e:
    raise e